<a href="https://colab.research.google.com/github/rmadatt/AI-Newton-StudyGroupAfrica/blob/main/AI_Newton_Elastic_Collision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
# Here is a sample experimental configuration and data generation script in Python for a 1D elastic collision. In frameworks like AI-Newton, structured configurations like this define the initial state of physical entities (such as mass, position, and velocity) so the system can track trajectories, simulate interactions, and ultimately discover conservation laws (like momentum and energy).
#Key Components Explained: (A)Object Properties: The configuration explicitly specifies independent variables like mass, initial positions, and velocities without hardcoding any underlying physical laws. This allows discovery frameworks to treat them as raw inputs. (B)Time-Series Output: The simulation loop outputs temporal tracking data (position and velocity over time), which serves as the dataset for symbolic regression and plausible reasoning modules to deduce conserved quantities.
# 1. Define the experimental configuration structure
elastic_collision_config = {
    "experiment_id": "elastic_collision_1d_001",
    "system_type": "two_body_collision",
    "parameters": {
        "time_step": 0.01,         # Simulation resolution (dt)
        "total_duration": 3.0      # Total duration of the experiment
    },
    "objects": [
        {
            "name": "Body_A",
            "mass": 2.0,           # Mass units
            "initial_position": 0.0, # Initial coordinate (m)
            "initial_velocity": 4.0  # Initial speed (m/s)
        },
        {
            "name": "Body_B",
            "mass": 3.0,
            "initial_position": 8.0,
            "initial_velocity": -1.0
        }
    ]
}

def run_elastic_collision_experiment(config):
    """
    Simulates the elastic collision experiment and returns
    time-series observation logs required for symbolic law discovery.
    """
    obj_a, obj_b = config["objects"]

    m1, x1, v1 = obj_a["mass"], obj_a["initial_position"], obj_a["initial_velocity"]
    m2, x2, v2 = obj_b["mass"], obj_b["initial_position"], obj_b["initial_velocity"]

    dt = config["parameters"]["time_step"]
    duration = config["parameters"]["total_duration"]
    steps = int(duration / dt)

    observation_logs = []
    collision_occurred = False

    for step in range(steps):
        t = step * dt

        # Detect collision point (when Body A catches up or meets Body B)
        if not collision_occurred and x1 >= x2:
            # 1D Elastic Collision analytical velocity update formulas
            v1_final = ((m1 - m2) * v1 + 2 * m2 * v2) / (m1 + m2)
            v2_final = ((m2 - m1) * v2 + 2 * m1 * v1) / (m1 + m2)
            v1, v2 = v1_final, v2_final
            collision_occurred = True

        # Update positions based on current velocities
        x1 += v1 * dt
        x2 += v2 * dt

        # Log observations at each time step
        observation_logs.append({
            "time": round(t, 3),
            "body_a_position": round(x1, 4),
            "body_a_velocity": round(v1, 4),
            "body_b_position": round(x2, 4),
            "body_b_velocity": round(v2, 4)
        })

    return observation_logs

# Execute the simulation
dataset = run_elastic_collision_experiment(elastic_collision_config)

# Display sample data points
print(f"Generated {len(dataset)} observation frames successfully.")
print("Initial State:", dataset[0])
print("Mid-simulation State (Near Collision):", dataset[150])
print("Post-collision State:", dataset[-1])

Generated 300 observation frames successfully.
Initial State: {'time': 0.0, 'body_a_position': 0.04, 'body_a_velocity': 4.0, 'body_b_position': 7.99, 'body_b_velocity': -1.0}
Mid-simulation State (Near Collision): {'time': 1.5, 'body_a_position': 6.04, 'body_a_velocity': 4.0, 'body_b_position': 6.49, 'body_b_velocity': -1.0}
Post-collision State: {'time': 2.99, 'body_a_position': 3.66, 'body_a_velocity': -2.0, 'body_b_position': 10.56, 'body_b_velocity': 3.0}


In [2]:
!pip install gplearn

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from gplearn.genetic import SymbolicRegressor

# Assuming `dataset` from the previous simulation is available:
# dataset contains keys: 'time', 'body_a_position', 'body_a_velocity', 'body_b_position', 'body_b_velocity'

# 1. Prepare data matrix for feature extraction
# Let's extract pre-collision (t = 0.0) and post-collision (t = 2.99) states
# to discover the velocity mapping function: v1_final = f(m1, m2, v1_init, v2_init)

mass_a = 2.0
mass_b = 3.0

# For demonstration, let's build a small training dataset of multiple collision variations
# by varying initial velocities to give the symbolic regressor enough variance to learn from.
np.random.seed(42)
n_samples = 200

v1_inits = np.random.uniform(1.0, 6.0, n_samples)
v2_inits = np.random.uniform(-4.0, -0.5, n_samples)

# Analytical ground truth used here solely to generate diverse training rows for the ML algorithm
v1_finals = ((mass_a - mass_b) * v1_inits + 2 * mass_b * v2_inits) / (mass_a + mass_b)

# Features: X matrix containing [v1_initial, v2_initial]
X = np.column_stack((v1_inits, v2_inits))
# Target: y vector containing the resulting velocity of Body A after collision
y = v1_finals

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Initialize the Symbolic Regressor (Scikit-Learn Compatible)
sr_model = SymbolicRegressor(
    population_size=2000,
    generations=20,
    stopping_criteria=0.0001,
    p_crossover=0.7,
    p_subtree_mutation=0.1,
    p_point_mutation=0.1,
    max_samples=0.9,
    verbose=1,
    random_state=42
)

# 3. Fit the model to discover the mathematical law
print("Searching for the underlying physical law via symbolic regression...")
sr_model.fit(X_train, y_train)

# 4. Output the discovered formula
print("\n--- Discovery Results ---")
print("Discovered Mathematical Expression:", sr_model._program)
print(f"Model R2 Score on Test Data: {sr_model.score(X_test, y_test):.4f}")

Searching for the underlying physical law via symbolic regression...
    |   Population Average    |             Best Individual              |
---- ------------------------- ------------------------------------------ ----------
 Gen   Length          Fitness   Length          Fitness      OOB Fitness  Time Left
   0    37.95          9089.09        3         0.334963         0.348037     56.43s
   1    16.87          6.04443        9         0.280017         0.301991      1.39m
   2    12.15          6.24769       19         0.223605         0.266065      1.68m
   3     8.98          3.64075        9         0.119194          0.12537     37.78s
   4     9.53          3.22116        9         0.116935         0.145703     25.48s
   5    11.39          4.48795        9         0.115647         0.157299     24.15s
   6    12.65          3.41675        9          0.11249         0.185709     27.09s
   7    10.05          3.89843        9         0.112719         0.183647     34.68s
   8  

In [4]:
!pip install gplearn numpy pandas scikit-learn

In [6]:
import numpy as np
from gplearn.genetic import SymbolicRegressor

# 1. Simulate multiple elastic collision trials to create a rich observational dataset
np.random.seed(42)
n_trials = 300

m1, m2 = 2.0, 3.0  # Masses of Body A and Body B

# Vary initial velocities across different experimental trials
v1_init = np.random.uniform(2.0, 6.0, n_trials)
v2_init = np.random.uniform(-4.0, -1.0, n_trials)

# Ground-truth outcome of 1D elastic collision (what nature/the physics engine outputs)
v1_final = ((m1 - m2) * v1_init + 2 * m2 * v2_init) / (m1 + m2)

# 2. Formulate the Feature Matrix (X) and Target (y)
# X[:, 0] = initial velocity of Body A, X[:, 1] = initial velocity of Body B
X = np.column_stack((v1_init, v2_init))
y = v1_final

# 3. Initialize Genetic Programming to discover the collision law
collision_law_discoverer = SymbolicRegressor(
    population_size=4000,
    generations=20,
    tournament_size=25,
    stopping_criteria=1e-5,
    p_crossover=0.7,
    p_subtree_mutation=0.15,
    p_point_mutation=0.1,
    verbose=1,
    random_state=42,
    function_set=('add', 'sub', 'mul', 'div')
)

# 4. Run the discovery process tabula rasa (from raw data alone)
print("--- Discovering Elastic Collision Law from Data ---")
collision_law_discoverer.fit(X, y)

# 5. Inspect what the AI found
print("\n--- Discovered Law ---")
print("Discovered Mathematical Formula:", collision_law_discoverer._program)

--- Discovering Elastic Collision Law from Data ---
    |   Population Average    |             Best Individual              |
---- ------------------------- ------------------------------------------ ----------
 Gen   Length          Fitness   Length          Fitness      OOB Fitness  Time Left
   0    37.44          14527.4        7         0.302403              N/A      1.33m
   1    17.16          9.58619       21         0.227869              N/A      1.37m
   2    13.20          5.55705       15         0.137974              N/A     55.71s
   3    10.02          4.57644       13        0.0813172              N/A     52.18s
   4     8.72          16.9927       13        0.0813172              N/A     52.44s
   5     9.83          4.11028       13        0.0813172              N/A     56.57s
   6    12.45          3.42001       17        0.0660511              N/A     43.29s
   7    15.30           10.797       19        0.0308931              N/A     42.03s
   8    14.14          